# Analysis for the Project: Debt-Based License Suspension

## The Main Questions:
- What are the reasons for license suspensions? 
- What % are not falling under safety related? 
- What % are debt based?
- How many people lose their license over nothing dangerous?


In [2]:
import numpy as np
import polars as pl
import pathlib

In [3]:
reinstatement_path = pathlib.Path.cwd().parent.joinpath('data', r'Reinstatement_information_Part_2 (1).xlsx')
reinstatement = pl.read_excel(reinstatement_path, engine='calamine')

reinstatement = reinstatement.select(
    pl.col("Year Reinstated").cast(pl.UInt16),
    pl.col("*").exclude("Year Reinstated"),
)

print(reinstatement.shape)
reinstatement.head()

(66610, 6)


Year Reinstated,Sanction Description,Term,Days Sus/Rev,Authority,Issuing Agency
u16,str,str,str,null,null
2020,"""NON-PAYMENT OF…","""Indefinite""","""1072""",null,null
2020,"""PROPERTY DAMAG…","""Indefinite""","""714""",null,null
2020,"""NRVC Withdrawa…","""Indefinite""","""1191""",null,null
2020,"""NRVC Withdrawa…","""Indefinite""","""924""",null,null
2020,"""94C35 __ PRESE…","""Definite""","""365""",null,null


In [4]:
reinstatement.null_count()

Year Reinstated,Sanction Description,Term,Days Sus/Rev,Authority,Issuing Agency
u32,u32,u32,u32,u32,u32
0,0,9,0,66610,66610


In [5]:
suspension_path = pathlib.Path.cwd().parent.joinpath('data', r'Suspension_information_Part_1 (1).xlsx')
suspension = pl.read_excel(suspension_path, engine='calamine')

suspension = suspension.select(
    pl.col("Year").cast(pl.UInt16),
    pl.col("*").exclude("Year"),
)

print(suspension.shape)
suspension.head()

(116841, 6)


Year,Description,Term,Days Sus/Rev,Authority,Issuing Agency
u16,str,str,str,null,null
2020,"""Operating Unde…","""Definite""","""2920""",null,null
2020,"""Operating Unde…","""Definite""","""730""",null,null
2020,"""Operating Unde…","""Definite""","""2920""",null,null
2020,"""Ignition Inter…","""Indefinite""","""INDEFINITE""",null,null
2020,"""Operating Unde…","""Definite""","""730""",null,null


In [6]:
suspension.null_count()

Year,Description,Term,Days Sus/Rev,Authority,Issuing Agency
u32,u32,u32,u32,u32,u32
0,0,1804,0,116841,116841


In [7]:
sus_reasons = suspension.unique('Description')['Description'].to_list()
print(len(sus_reasons))
sus_reasons

138


['ELECTRONIC MESSAGE, OPERATOR SEND/READ * c9',
 'USE MV WITHOUT AUTHORITY, 3RD OFF. c90 §24(2)(a)',
 'LICENSE RESTRICTION, OPERATE MV IN VIOL c90 §10',
 'B08 - LVSC: property damage',
 'MOTOR VEH HOMICIDE OUI--DRUGS & NEGLIG c90 §24G(a)',
 'RMV DOCUMENT, POSSESS/USE FALSE/STOLEN c90 §24B',
 'M80 - Inatt/careless/neg drvin',
 'B05 - Leave scne before police',
 'INTERLOCK EARLY WITHDRAWAL',
 'Chemical Test Refusal',
 'MOBILE PHONE, OPERATOR UNDER 18 USE * c90 §8M',
 'LEAVE SCENE OF PERSONAL INJURY & DEATH c90 §24(2)(a½)(2)',
 'MOTOR VEH VIN, REMOVE/ALTER c266 §139(a)',
 'JUNIOR OPERATOR OP 12-5 AM W/O PARENT c90 §§8 & 10',
 'M81 - Careless/imprper driving',
 'D36 - Failure to maintain liability insurance',
 'LICENSE SUSPENDED, OP MV WITH c90 §23',
 'Three Speeding Offenses',
 'RACING MOTOR VEHICLE BY LICENSED OPERATOR c90 §17B',
 'S93 - Speeding',
 'USE MV WITHOUT AUTHORITY c90 §24(2)(a)',
 'B56 - Driving CMV w/o CDL',
 'NEGLIGENT OPERATION & INJURY FROM MOBILE PHONE USE c90 §24(2)(a)',

In [8]:
cur_suspended_path = pathlib.Path.cwd().parent.joinpath('data', r'Currently_Suspended_information_Part_3_as_of_12082_022 (1).xlsx')
cur_suspended = pl.read_excel(cur_suspended_path, engine='calamine')
print(cur_suspended.shape)
cur_suspended.head()

(693100, 5)


Description,Term,Days Sus/Rev,Authority,Issuing Agency
str,str,str,null,null
"""COURT DEFAULT""","""Indefinite""","""INDEFINITE""",null,null
"""W00 - Non-ACD …","""Indefinite""","""INDEFINITE""",null,null
"""Ignition Inter…","""Indefinite""","""INDEFINITE""",null,null
"""W00 - Non-ACD …","""Indefinite""","""INDEFINITE""",null,null
"""FSU __ 5 SURCH…","""Indefinite""","""INDEFINITE""",null,null


In [9]:
cur_suspended.null_count()

Description,Term,Days Sus/Rev,Authority,Issuing Agency
u32,u32,u32,u32,u32
0,10927,0,693100,693100


In [10]:
default = cur_suspended.filter((pl.col("Description") == "PAYMENT DEFAULT") & (pl.col("Term") == "Indefinite"))
print(f'{default.shape[0]} suspensions due to payment default, {round(default.shape[0]/cur_suspended.shape[0]*100, 3)}% of {cur_suspended.shape[0]} total suspensions')

94841 suspensions due to payment default, 13.684% of 693100 total suspensions
